# Step 2: CSCI (ICCV 2025) features on MEVID

Runs the authors' **released MEVID video checkpoint** (EVA-02-L + EZ-CLIP, color profile 17) with their own evaluation code, then saves the features in the same format as the OSNet runs, so every downstream analysis (accuracy, audit rule, consistent ID) works unchanged.

**Checkpoint 1:** their evaluation should print about **79.7% top-1 / 56.7% mAP** (overall). If it does, the backbone is reproduced.

Only the **test** split images are needed. Run all cells in order on an A100.

In [ ]:
# 1. Settings, Drive mount, find the MEVID folder automatically
import os, glob, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')

SHARED = os.path.join('/content/drive/Shareddrives', 'YOUR_SHARED_DRIVE', 'Research')
KNOWN = [os.path.join(SHARED, 'Data', 'MEVID'),
         os.path.join(SHARED, 'DATA', 'MEVID'),
         os.path.join('/content/drive/MyDrive', 'MEVID')]
TARGET = 'mevid-v1-bbox-test.tgz'

def find_mevid():
    for p in KNOWN:
        if os.path.isfile(os.path.join(p, TARGET)):
            return p
    print('Not in the known locations; searching Drive (can take a minute) ...')
    roots = glob.glob('/content/drive/Shareddrives/*') + ['/content/drive/MyDrive']
    for root in roots:
        for depth in range(0, 5):
            pattern = os.path.join(root, *(['*'] * depth), TARGET)
            hits = glob.glob(pattern)
            if hits:
                return os.path.dirname(hits[0])
    return None

DATA_DIR = find_mevid()
if DATA_DIR is None:
    sd = '/content/drive/Shareddrives'
    print('Shared drives visible:', os.listdir(sd) if os.path.isdir(sd) else 'NONE')
    if os.path.isdir(SHARED):
        print('Folders in Research:', sorted(os.listdir(SHARED)))
    raise FileNotFoundError(TARGET + ' not found on Drive. Paste the lines above.')

print('Data dir  :', DATA_DIR)

# Results go under the same Research folder the data lives in (whatever the
# shared drive is actually called), else next to the MEVID folder.
if '/Research/' in DATA_DIR:
    research_root = DATA_DIR.split('/Research/')[0] + '/Research'
    OUTPUT_DIR = os.path.join(research_root, '2026', 'MEVID', 'OSNet-baseline')
else:
    OUTPUT_DIR = os.path.join(os.path.dirname(DATA_DIR), 'MEVID-results', 'OSNet-baseline')
LOCAL_DIR = '/content/mevid'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

# Frozen baseline values (OSNet baseline)
REID_GLOBAL_FLOOR = 0.65
REID_THETA_CAP    = 0.80
REID_MARGIN       = 0.05

MAX_FRAMES_PER_TRACK = None   # None = every frame. Set e.g. 64 for speed.
BATCH_SIZE  = 512
NUM_WORKERS = min(12, os.cpu_count() or 2)

print('Contents  :', sorted(os.listdir(DATA_DIR)))
print('Output dir:', OUTPUT_DIR)

In [ ]:
# 2. Test images + annotations on local disk; build the folder layout CSCI expects
import sys
def find_one(pattern):
    hits = glob.glob(os.path.join(DATA_DIR, '**', pattern), recursive=True)
    assert hits, f'{pattern} not found under {DATA_DIR}'
    return hits[0]

CSCI_OUT = os.path.join(os.path.dirname(OUTPUT_DIR), 'CSCI')
os.makedirs(CSCI_OUT, exist_ok=True)

marker = f'{LOCAL_DIR}/.done_test'
if not os.path.exists(marker):
    if os.path.isdir(f'{LOCAL_DIR}/bbox_test'):
        shutil.rmtree(f'{LOCAL_DIR}/bbox_test')
    tgz = find_one('mevid-v1-bbox-test.tgz')
    print('Extracting', tgz, '(about 10 minutes)')
    subprocess.run(['tar', '-xzf', tgz, '-C', LOCAL_DIR], check=True)
    open(marker, 'w').close()
subprocess.run(['unzip', '-oq', find_one('mevid-v1-annotation-data.zip'),
                '-d', LOCAL_DIR], check=True)

# CSCI expects ROOT/{train_name,test_name,track_*_info,query_IDX}.txt and
# ROOT/bbox_test/<pid>/<image>.jpg
ROOT = '/content/mevid_csci'
os.makedirs(ROOT, exist_ok=True)
for n in ['train_name.txt', 'test_name.txt', 'track_train_info.txt',
          'track_test_info.txt', 'query_IDX.txt']:
    src = sorted(glob.glob(f'{LOCAL_DIR}/**/{n}', recursive=True))[0]
    dst = f'{ROOT}/{n}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
for split in ['bbox_test', 'bbox_train']:
    if os.path.isdir(f'{LOCAL_DIR}/{split}') and not os.path.exists(f'{ROOT}/{split}'):
        os.symlink(f'{LOCAL_DIR}/{split}', f'{ROOT}/{split}')
print('CSCI root:', ROOT, sorted(os.listdir(ROOT)))
print('Outputs  :', CSCI_OUT)

In [ ]:
# 3. CSCI code, dependencies, released MEVID weights, and a feature-dump hook
REPO = '/content/ICCV-CSCI-Person-ReID'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/ppriyank/ICCV-CSCI-Person-ReID.git', REPO],
                   check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.9', 'einops', 'yacs', 'mmengine', 'decord',
                'tensorboardX', 'tensorboard', 'termcolor', 'tabulate', 'h5py',
                'scikit-learn', 'lmdb', 'albumentations', 'gdown', 'pandas'],
               check=True)

WT = f'{CSCI_OUT}/ez_eva02_vid_hybrid_extra_best.pth'      # cached on Drive
if not os.path.exists(WT):
    url = ('https://huggingface.co/ppriyank/CSCI-CC-ReID/resolve/main/'
           'mevid-17-1244/ez_eva02_vid_hybrid_extra_best.pth')
    subprocess.run(['wget', '-q', '-O', WT, url], check=True)
print('Weights:', WT, f'{os.path.getsize(WT) / 1e9:.2f} GB')

# Hook: save query/gallery features right after CSCI computes them
f = f'{REPO}/processor/eval_mevid.py'
src = open(f).read()
anchor = "    distmat = distmat.cpu()\n"
hook = (anchor +
        "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
        "        import numpy as _np\n"
        "        _np.savez(os.environ['CSCI_DUMP'], qf=qf.float().cpu().numpy(),"
        " gf=gf.float().cpu().numpy(), q_pids=q_pids.numpy(),"
        " g_pids=g_pids.numpy(), q_camids=q_camids.numpy(),"
        " g_camids=g_camids.numpy())\n"
        "        logger.info('Dumped features to ' + os.environ['CSCI_DUMP'])\n")
if 'CSCI_DUMP' not in src:
    assert src.count(anchor) == 1, 'eval_mevid.py changed upstream; tell Claude'
    src = src.replace(anchor, hook)
    if '\nimport os' not in src:
        src = 'import os\n' + src
    open(f, 'w').write(src)
print('Feature-dump hook installed')

In [ ]:
# 4. Run CSCI's own MEVID evaluation (reproduces their numbers + dumps features)
DUMP = f'{CSCI_OUT}/csci_mevid_test_features.npz'
TEST_BATCH = 32          # lower to 16 if you see CUDA out-of-memory
env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO)
cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12355',
       'train_two_step.py', '--eval', '--resume',
       '--config_file', 'configs/mevid_eva02_l_cloth.yml',
       'DATA.ROOT', ROOT + '/', 'TEST.WEIGHT', WT, 'SOLVER.SEED', '12345',
       'TRAIN.TRAIN_VIDEO', 'True', 'DATA.DATASET', 'mevid',
       'MODEL.NAME', 'ez_eva02_vid_hybrid_extra', 'TRAIN.COLOR_PROFILE', '17',
       'DATA.TEST_BATCH', str(TEST_BATCH)]
log_path = f'{CSCI_OUT}/csci_eval_log.txt'
with open(log_path, 'w') as log:
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        log.write(line)
        if any(k in line for k in ['top1', 'mAP', 'Error', 'error', 'Dumped',
                                   'Traceback', 'Extract', 'loaded']):
            print(line.rstrip())
    p.wait()
print('Exit code:', p.returncode, '| full log:', log_path)
assert p.returncode == 0 and os.path.exists(DUMP), 'CSCI eval failed; paste the last 40 lines of the log'
print(''.join(open(log_path).readlines()[-40:]))

In [ ]:
# 5. Convert to the same format as the OSNet runs and check alignment
import numpy as np, re
z = np.load(DUMP)
info = np.loadtxt(f'{ROOT}/track_test_info.txt').astype(int)
qidx = np.loadtxt(f'{ROOT}/query_IDX.txt').astype(int)
gidx = np.array([i for i in range(len(info)) if i not in set(qidx)])

assert len(z['qf']) == len(qidx) and len(z['gf']) == len(gidx)
assert (z['q_pids'] == info[qidx, 2]).all(), 'query order mismatch'
assert (z['g_pids'] == info[gidx, 2]).all(), 'gallery order mismatch'

D = z['qf'].shape[1]
te_T = np.zeros((len(info), D), np.float32)
te_T[qidx], te_T[gidx] = z['qf'], z['gf']
te_T /= np.maximum(np.linalg.norm(te_T, axis=1, keepdims=True), 1e-8)
te_pid, te_oid, te_cam = info[:, 2], info[:, 3], info[:, 4]
np.savez(f'{CSCI_OUT}/test_templates_csci.npz', templates=te_T,
         pid=te_pid, oid=te_oid, cam=te_cam, qidx=qidx, gidx=gidx)

# Quick independent check: overall Rank-1 / mAP with our own scorer
def ap_cmc(order, good, junk):
    order = order[~np.isin(order, junk)]
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc

sim = te_T[qidx] @ te_T[gidx].T
APs, R1 = [], []
for i, q in enumerate(qidx):
    sp = np.flatnonzero(te_pid[gidx] == te_pid[q])
    sc = np.flatnonzero(te_cam[gidx] == te_cam[q])
    good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
    if good.size == 0:
        continue
    ap, cmc = ap_cmc(np.argsort(-sim[i]), good, junk)
    APs.append(ap); R1.append(cmc[0])
print(f'Features: {te_T.shape} | our scorer, overall: '
      f'mAP {100*np.mean(APs):.1f}  Rank-1 {100*np.mean(R1):.1f}')
print('Saved', f'{CSCI_OUT}/test_templates_csci.npz')